# 26. 数据质量检查与清洗

<!-- module-learning-arc:start -->
> **Pandas 模块主线｜第 5 / 10 步：修正类型并建立干净字段**
>
> **持续应用背景：** 搭建电商履约异常追踪台：把订单、客户、商品和履约信息整理成安全合并的事实表，再生成趋势指标和异常工单。
>
> **承接上一阶段：** 行列操作与类型转换  →  **本章任务：** 数据质量检查与清洗  →  **下一步：** 文本、日期与特征处理
>
> **大作业连接：** 本章练习将成为《电商履约异常追踪台》的一部分，最终需要从多表质量审计走到订单粒度事实表、窗口趋势和可复核异常工单。
<!-- module-learning-arc:end -->


## 本章场景

分析的第一步不是套模型，而是先确认数据“能不能用”——有没有缺失、重复、数值异常。



## 本章目标

学完本章，你将能够：

- **理解**：理解缺失值/重复值/异常值的识别与处理。
- **操作**：能检查并清洗缺失、重复、类型不一致的脏数据。
- **迁移**：能对一份有脏数据的经营明细做出可复核的清洗口径与处理。


## 26.1 核心概念

**背景引入**：分析的第一步不是套模型，而是先确认数据“能不能用”——有没有缺失、重复、数值异常。这些脏数据如果直接进分析，缺一个值会算错汇总，重复一行会放大统计量，一个异常大额单又会带偏均值。学会先做质量检查与清洗，你的每一步后续结论才站得住脚，也能少踩“结果对不上账”的坑。

- 先统计问题规模，再决定删除、填充或保留。
- 缺失值处理取决于业务含义，不能统一填0。
- 异常值应先标记和调查，不能机械删除。

> **直观类比**：缺失值像报表里没填的格子，“没发生”和“忘了填/不该有”是两回事；一律填 0，就好比把“空房租”和“租户跑路”都记成“空房”，账面会失真。先问“为什么缺”，再决定填、删还是保留。


## 26.2 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| isna() | `pd.DataFrame()`、`data.isna()`、`.sum()`、`sum()` | 先统计缺失规模，再决定删除还是填充。 | 看到缺失值就全部填0 |
| fillna() | `pd.DataFrame()`、`.fillna()`、`data['region']` | 填充值必须符合字段含义，不能看到缺失就统一填0。 | 删除重复时未说明唯一键 |
| dropna() | `pd.DataFrame()`、`data.dropna()` | 只有在缺失记录确实无法参与分析时才删除。 | 把真实的大额订单误判为错误数据 |
| duplicated() 与 drop_duplicates() | `pd.DataFrame()`、`data.duplicated()`、`data.drop_duplicates()`、`.sum()` | 先统计重复，再按明确业务键去重。 | 看到缺失值就全部填0 |
| quantile() 异常阈值 | `pd.Series()`、`values.quantile()`、`.tolist()`、`values[values > upper]` | IQR规则适合标记潜在异常，但不等于直接删除。 | 删除重复时未说明唯一键 |


## 26.3 示例 1：质量概览

**背景引入**：拿到一张新表，最怕的不是数据难看，而是盯着数字往下做，做到一半才发现缺了一角、重了一行。动手分析前先用几行代码把"家底"摸清——形状、缺失、重复、类型，哪里有问题心里先有数，后面才不会让脏数据带偏结论。

**讲解**：从四个角度同时给一张表做"体检"，每个角度对应一种检查方法。

- `orders.shape` 看形状：几行几列，先确认数据规模对不对；
- `orders.isna().sum()` 看缺失：逐列数空值数量，决定要不要处理；
- `orders.duplicated().sum()` 看重复：统计完全相同的整行有几条；
- `orders.dtypes` 看类型：金额是不是数值、字段是不是被读成了文本；
- **口诀**：先看形状后看缺，重复类型一起查，体检过关再干活。


<!-- math-foundation:chapter-26 -->
### 数学推导｜把数据质量问题量化

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜为每个检查单位建立标记。** 缺失标记 $m_i$、重复标记 $u_i$ 都只取 0 或 1。

$$
m_i=\mathbf{1}(x_i\text{ 缺失}),\qquad u_i=\mathbf{1}(key_i\text{ 重复})
$$

**第 2 步｜0/1 均值就是比例。** 因为 $\sum_i m_i=N_{miss}$，所以

$$
\frac{1}{N}\sum_i m_i=\frac{N_{miss}}{N}
$$

重复率同理。关键不是公式本身，而是分母 $N$ 到底表示行、主键还是单元格。

**把上面的关系收束为本章计算式：**

$$
r_{miss}=\frac{N_{miss}}{N},\qquad r_{dup}=\frac{N_{dup}}{N}
$$

**符号解释：** $N$ 是检查单位总数，$N_{miss}$、$N_{dup}$ 分别是缺失和重复数量。

**代码对应：** `df.isna().mean()` 计算列缺失率，`df.duplicated(key).mean()` 计算重复率。

**使用边界：** 分母必须与检查粒度一致；单元格缺失率、行缺失率和主键缺失率不能混用。


In [ ]:
import numpy as np
import pandas as pd

orders = pd.DataFrame(
    {
        "order_id": ["A1", "A2", "A2", "A3", "A4"],
        "region": ["华东", "华南", "华南", None, "华北"],
        "amount": [320.0, 880.0, 880.0, np.nan, 9800.0],
    }
)
print("形状:", orders.shape)
print("缺失:\n", orders.isna().sum())
print("重复行:", orders.duplicated().sum())
print(orders.dtypes)


## 26.4 示例 2：缺失与重复处理

**背景引入**：体检发现问题就要动手修。这张表里 A2 订单出现两次，是录入重复还是两次下单？还有地区、金额的缺失——直接删还是补，取决于字段的业务含义，不能随手全填 0。

**讲解**：按"先定唯一键去重，再按含义填缺失"两步处理，规则写清楚、口径讲明白。

- `drop_duplicates(subset="order_id", keep="first")` 按订单号去重，明确保留第一条；
- `region` 是地区字段，缺失用 `fillna("未知")` 补文字，不能填数字 0；
- `amount` 是金额字段，缺失用 **中位数** `median()` 填充，比填 0 更贴近真实水平；
- **口诀**：去重先定唯一键，缺失填充看含义，地区补字金额补中位。


In [ ]:
clean = orders.drop_duplicates(subset="order_id", keep="first").copy()
clean["region"] = clean["region"].fillna("未知")
median_amount = clean["amount"].median()
clean["amount"] = clean["amount"].fillna(median_amount)
print(clean)


## 26.5 示例 3：IQR异常标记

**背景引入**：9800 元这一单，是真实的大额客户还是录入错误？不能凭感觉直接删——万一是真金白银的大订单呢。稳妥的做法是先把它"标记"出来、保留原值，再交给业务去调查，而不是机械地当错误数据清掉。

**讲解**：用 IQR（四分位距）规则算出一个合理上界，`amount > upper` 的那批就是潜在异常，用新列布尔值标出、原数据不动。

- `quantile(0.25)` 和 `quantile(0.75)` 分别取下四分位、上四分位；
- `iqr = q3 - q1`，上界 `upper = q3 + 1.5 * iqr`，超过上界即视为异常；
- `clean["is_outlier"]` 新增布尔列做标记，不删除原值、方便后续追查；
- **口诀**：IQR 定上界，超界标成异常值，先标记再调查，别急着丢掉。


In [ ]:
q1 = clean["amount"].quantile(0.25)
q3 = clean["amount"].quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr
clean["is_outlier"] = clean["amount"] > upper
print("上界:", upper)
print(clean[clean["is_outlier"]])


## 26.6 核心操作独立示例

下面每个代码单元格只演示一个核心方法、函数或语法操作。请先阅读方法名称和任务说明，再单独运行当前单元格；示例尽量自带最小输入，不要求依赖前一个单元格留下的变量。


In [ ]:
# isna()
# 先统计缺失规模，再决定删除还是填充。
import numpy as np
import pandas as pd

data = pd.DataFrame(
    {"region": ["华东", None, "华北"], "amount": [320, np.nan, 460]}
)
print(data.isna())
print(data.isna().sum())


In [ ]:
# fillna()
# 填充值必须符合字段含义，不能看到缺失就统一填0。
import pandas as pd

data = pd.DataFrame({"region": ["华东", None], "amount": [320, 880]})
data["region"] = data["region"].fillna("未知")
print(data)


In [ ]:
# dropna()
# 只有在缺失记录确实无法参与分析时才删除。
import pandas as pd

data = pd.DataFrame({"id": ["A1", "A2", "A3"], "amount": [320, None, 460]})
clean = data.dropna(subset=["amount"])
print(clean)


In [ ]:
# duplicated() 与 drop_duplicates()
# 先统计重复，再按明确业务键去重。
import pandas as pd

data = pd.DataFrame(
    {"customer_id": ["U1", "U2", "U2"], "amount": [320, 880, 880]}
)
print("重复数:", data.duplicated(subset=["customer_id"]).sum())
print(data.drop_duplicates(subset=["customer_id"]))


In [ ]:
# quantile() 异常阈值
# IQR规则适合标记潜在异常，但不等于直接删除。
import pandas as pd

values = pd.Series([10, 12, 13, 15, 18, 20, 100])
q1, q3 = values.quantile([0.25, 0.75])
upper = q3 + 1.5 * (q3 - q1)
print("上界:", upper)
print("潜在异常:", values[values > upper].tolist())


**练一练 22.6**：下面是一张含问题的订单表，请完成三步：① 用 `isna()` 统计每个字段的缺失数；② 按 `order_id` 去重；③ 用 `fillna()` 补齐缺失（`region` 填 `"未知"`、`amount` 填该列中位数），最后确认整张表已无缺失。

```python
import numpy as np
import pandas as pd
orders = pd.DataFrame({
    "order_id": ["B1", "B2", "B2", "B3"],
    "region":  ["华东", None, "华南", "华北"],
    "amount":  [200.0, 450.0, 450.0, np.nan],
})
```


In [ ]:
# 请在下方填写代码
import numpy as np
import pandas as pd

# TODO: ① 统计每个字段的缺失数
# TODO: ② 按 order_id 去重（保留第一条）
# TODO: ③ 填充缺失并确认无缺失
# TODO：请在下方完成 —— 练一练 22.6：下面是一张含问题的订单表，请完成三步：① 用 isna() 统计每个字段的缺失数；② 按 order_


In [ ]:
import numpy as np
import pandas as pd

orders = pd.DataFrame(
    {
        "order_id": ["B1", "B2", "B2", "B3"],
        "region": ["华东", None, "华南", "华北"],
        "amount": [200.0, 450.0, 450.0, np.nan],
    }
)

# ① 统计缺失
missing = orders.isna().sum()
print("各字段缺失数:\n", missing)

# ② 按 order_id 去重
clean = orders.drop_duplicates(subset=["order_id"]).copy()
print("去重后行数:", len(clean))

# ③ 填充缺失
clean["region"] = clean["region"].fillna("未知")
clean["amount"] = clean["amount"].fillna(clean["amount"].median())
print(clean)
print("剩余缺失总数:", clean.isna().sum().sum())


## 26.7 公开大型数据实战

下面使用 UCI Machine Learning Repository 的 Online Retail 公开数据集。原始数据包含 541,909 条英国在线零售交易，本课程使用固定随机种子抽取的 200,000 行子集。分析时在完整子集上计算，只展示摘要或少量样本。


In [ ]:
import numpy as np
import pandas as pd

# UCI Machine Learning Repository: Online Retail
# 原始数据 541,909 行；课程使用固定随机种子抽取的 200,000 行子集。
data_url = "/datasets/uci_online_retail_200k.csv"
large_orders = pd.read_csv(
    data_url,
    parse_dates=["InvoiceDate"],
    dtype={
        "InvoiceNo": "string",
        "StockCode": "string",
        "Description": "string",
        "Country": "category",
    },
).rename(
    columns={
        "InvoiceNo": "order_id",
        "StockCode": "stock_code",
        "Description": "description",
        "Quantity": "quantity",
        "InvoiceDate": "order_time",
        "UnitPrice": "unit_price",
        "CustomerID": "customer_id",
        "Country": "country",
    }
)
large_orders["sales"] = (
    large_orders["quantity"] * large_orders["unit_price"]
).round(2)
large_orders["status"] = np.where(
    large_orders["order_id"].str.startswith("C")
    | (large_orders["quantity"] < 0),
    "取消/退货",
    "完成",
)
print("UCI Online Retail 公开数据：")
print(f"  {len(large_orders):,} 行 × {large_orders.shape[1]} 列")
print(
    "内存占用：", f"{large_orders.memory_usage(deep=True).sum() / 1024**2:.1f} MB"
)
large_orders.head()


In [ ]:
quality = pd.DataFrame(
    {
        "缺失数": large_orders.isna().sum(),
        "缺失率": large_orders.isna().mean(),
        "唯一值": large_orders.nunique(dropna=False),
    }
).sort_values("缺失率", ascending=False)
print("完全重复行：", large_orders.duplicated().sum())
print("取消/退货行：", (large_orders["status"] == "取消/退货").sum())
display(quality.head(8))
clean_orders = (
    large_orders.drop_duplicates()
    .query("quantity > 0 and unit_price > 0")
    .dropna(subset=["description"])
)
print(f"清洗后保留：{len(clean_orders):,} / {len(large_orders):,} 行")


## 26.8 独立迁移练习

替换一个字段或分组口径，并核对处理前后的行数与粒度。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# TODO: 在此粘贴或改写最接近的示例。
# 记录：我改了什么？预期会发生什么？实际观察到什么？
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print({"修改": change_note, "预期": expected_change, "观察": observed_change})


## 26.9 本章实训：分组汇总与粒度

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd

orders = pd.DataFrame(
    {
        "region": ["华东", "华东", "华南", "华南"],
        "channel": ["线上", "线下", "线上", "线下"],
        "sales": [120, 80, 150, 100],
    }
)
summary = orders.groupby("region", as_index=False)["sales"].sum()
print(summary)
print("汇总表每一行代表一个地区")


### 26.9.1 第一个结果怎么读

先确认明细表一行代表一笔订单，再确认汇总表一行代表一个地区。`groupby` 的字段决定结果的粒度。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。



In [ ]:
orders["sales_level"] = orders["sales"].map(
    lambda value: "高" if value >= 120 else "普通"
)
print(orders)
print(orders["sales_level"].value_counts())


### 26.9.2 第二个结果怎么读

第二个实验只增加一个分类列，不改变原始销售额。练习解释：什么时候应该新增列，什么时候应该直接筛选行？

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。



## 26.10 错误恢复：脏数据转换怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

raw = pd.Series(["12", "unknown", "18", ""])
converted = pd.to_numeric(raw, errors="coerce")
print("转换结果：")
print(converted)
print("无法转换的数量：", converted.isna().sum())
print("后续可以选择删除、填充或回查原始值。")


### 26.10.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

errors="coerce" 会把无法转换的值记录为缺失，适合先完成质量盘点；不要在没有统计数量前直接删除。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。



## 26.11 易错点提醒

- 看到缺失值就全部填0
- 删除重复时未说明唯一键
- 把真实的大额订单误判为错误数据


## 26.12 练习与作业

1. 创建含缺失和重复的客户表
2. 按客户ID去重
3. 使用中位数填充年龄并输出质量报告

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 26.13 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“创建含缺失和重复的客户表”。
2. **独立完成**：不复制示例代码，完成“按客户ID去重”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“使用中位数填充年龄并输出质量报告”，用一两句话说明你修改了什么。

### 26.13.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 26.13.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
import numpy as np
import pandas as pd

# TODO: 按客户ID去重
# TODO: 使用中位数填充年龄
# TODO: 填充城市缺失值为"未知"
# TODO：请在下方完成 —— 22.13 练习与作业 1. 创建含缺失和重复的客户表 2. 按客户ID去重 3. 使用中位数填充年龄并输出质量报告 提


In [ ]:
import numpy as np
import pandas as pd

customers = pd.DataFrame(
    {
        "customer_id": ["U1", "U2", "U2", "U3"],
        "age": [28, np.nan, np.nan, 42],
        "city": ["上海", "广州", "广州", None],
    }
)
clean = customers.drop_duplicates("customer_id").copy()
clean["age"] = clean["age"].fillna(clean["age"].median())
clean["city"] = clean["city"].fillna("未知")
print(clean)
print(clean.isna().sum())


## 26.14 小结

建立数据质量检查流程，处理缺失、重复、异常和无效记录。

**迁移思考**：

1. 如果一个订单表中订单ID不重复，但同一用户有多个订单，去重时应该用什么键？
2. 为什么异常值应该先标记而不是直接删除？什么情况下可以删除异常值？



### 26.14.1 你已经掌握

- 生成质量概览
- 处理缺失值
- 识别并删除重复
- 使用规则标记异常



### 26.14.2 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。



### 26.14.3 需要注意

- 看到缺失值就全部填0
- 删除重复时未说明唯一键
- 把真实的大额订单误判为错误数据



### 26.14.4 完成检查

- [ ] 能够生成质量概览
- [ ] 能够处理缺失值
- [ ] 能够识别并删除重复
- [ ] 能够使用规则标记异常



### 26.14.5 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。

